# Training Infrastructure & Distributed Basics

MLEs at scale need to reason about how training is parallelized, how memory is managed, and what to checkpoint. This note covers parallelism strategies, DDP with a NumPy simulation, gradient accumulation, and GPU memory math.

## What Interviewers Test
- Data parallelism vs model parallelism vs pipeline parallelism — when each is used
- What AllReduce does in DDP and why it works
- Mixed precision: fp16, bf16, loss scaling
- Gradient accumulation as a memory-batch-size tradeoff
- GPU memory math: parameters + optimizer states + activations
- Checkpointing: what to save, how to resume

## Parallelism Strategies

| Strategy | What it parallelizes | When to use | Communication pattern |
|---|---|---|---|
| **Data parallelism (DDP)** | Different data batches on each GPU | Model fits in one GPU | AllReduce gradients every step |
| **Model parallelism** | Different layers on different GPUs | Model too large for one GPU | P2P activation passing |
| **Pipeline parallelism** | Micro-batches through pipeline of GPUs | Deep models, many layers | P2P, bubble overhead |
| **Tensor parallelism** | Single layer split across GPUs (e.g., Megatron) | Very large transformer layers | AllReduce within each layer |
| **ZeRO (DeepSpeed)** | Shard optimizer states, gradients, params | LLM training | AllGather/ReduceScatter |


In [ ]:
import numpy as np

np.random.seed(42)

# --- DDP AllReduce simulation ---
def simulate_ddp_allreduce(n_gpus=4, n_params=1000):
    """
    In DDP, each GPU computes gradients on its local batch.
    AllReduce averages gradients across all GPUs so every GPU has the same gradient.
    This is the ring-AllReduce pattern: O(N * p/G) per GPU, not O(N*p).
    """
    # Each GPU has a different batch → different local gradients
    local_grads = [np.random.randn(n_params) for _ in range(n_gpus)]
    
    print(f"=== DDP AllReduce Simulation ({n_gpus} GPUs) ===")
    print(f"Before AllReduce — gradient variance per GPU:")
    for i, g in enumerate(local_grads):
        print(f"  GPU {i}: mean={g.mean():.4f}, std={g.std():.4f}")
    
    # AllReduce: average across GPUs (in practice: ring-based)
    avg_grad = np.mean(local_grads, axis=0)
    
    print(f"\nAfter AllReduce — all GPUs get same gradient:")
    print(f"  Averaged: mean={avg_grad.mean():.4f}, std={avg_grad.std():.4f}")
    print(f"  (std reduced by ~1/sqrt({n_gpus}) = {1/n_gpus**0.5:.3f} due to averaging)")
    
    return avg_grad

avg = simulate_ddp_allreduce(n_gpus=4, n_params=500)


In [ ]:
# --- GPU memory math ---
def gpu_memory_estimate(
    n_params,
    bytes_per_param=2,          # fp16 = 2 bytes, fp32 = 4 bytes
    optimizer='adam',            # 'sgd': 1 copy, 'adam': 2 extra copies (m, v)
    batch_size=32,
    seq_len=512,
    n_layers=12,
    d_model=768,
    store_activations=True,
    bytes_gb=1e9,
):
    # Model parameters
    param_mem = n_params * bytes_per_param
    
    # Gradients (same size as params, fp16 if AMP)
    grad_mem = n_params * bytes_per_param
    
    # Optimizer states (fp32 for numerical stability in Adam)
    opt_states = {'sgd': 0, 'adam': 2}[optimizer] * n_params * 4  # fp32
    
    # Activations (rough estimate for transformer: O(batch * seq * d_model * n_layers))
    act_mem = batch_size * seq_len * d_model * n_layers * 4 if store_activations else 0
    
    total = param_mem + grad_mem + opt_states + act_mem
    return {
        'params_GB':       param_mem / bytes_gb,
        'gradients_GB':    grad_mem  / bytes_gb,
        'optimizer_GB':    opt_states / bytes_gb,
        'activations_GB':  act_mem  / bytes_gb,
        'total_GB':        total    / bytes_gb,
    }

# GPT-2 small: 117M params
print("=== GPT-2 Small (117M params) ===")
mem = gpu_memory_estimate(n_params=117e6, optimizer='adam',
                          batch_size=8, seq_len=1024, n_layers=12, d_model=768)
for k, v in mem.items():
    print(f"  {k:<20}: {v:.2f} GB")

print()
# LLaMA-7B: 7B params  
print("=== LLaMA-7B (7B params, fp16) ===")
mem_llama = gpu_memory_estimate(n_params=7e9, bytes_per_param=2, optimizer='adam',
                                batch_size=1, seq_len=2048, n_layers=32, d_model=4096)
for k, v in mem_llama.items():
    print(f"  {k:<20}: {v:.2f} GB")
print("  (A100 80GB: param+grad+optimizer = ~112GB → needs model parallelism or ZeRO)")


In [ ]:
import torch
import torch.nn as nn

# --- Gradient accumulation in PyTorch training loop ---
def training_loop_with_gradient_accumulation(model, dataloader_sim, optimizer,
                                              accumulate_steps=4):
    """
    Gradient accumulation: simulate large batch with small per-step batch.
    Effective batch size = per_step_batch * accumulate_steps.
    Loss must be divided by accumulate_steps so final gradient magnitude matches.
    """
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = model.to(device)
    
    step = 0
    optimizer.zero_grad()
    
    for micro_step, (X_batch, y_batch) in enumerate(dataloader_sim):
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        
        out = model(X_batch)
        loss = nn.functional.cross_entropy(out, y_batch)
        loss = loss / accumulate_steps   # CRITICAL: scale loss before backward
        loss.backward()                  # accumulate gradients
        
        if (micro_step + 1) % accumulate_steps == 0:
            optimizer.step()
            optimizer.zero_grad()
            step += 1
            if step <= 3:
                print(f"  Global step {step}: effective batch = {X_batch.shape[0] * accumulate_steps}")

# Simulate
model = nn.Sequential(nn.Linear(16, 32), nn.ReLU(), nn.Linear(32, 4))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def make_batch(n=8):
    return torch.randn(n, 16), torch.randint(0, 4, (n,))

dataloader_sim = [make_batch() for _ in range(20)]
print("Gradient accumulation (accumulate_steps=4, micro_batch=8 → effective batch=32):")
training_loop_with_gradient_accumulation(model, dataloader_sim[:12], optimizer, accumulate_steps=4)


## Common Interview Questions

**Q: What is DDP and how does it work?**
Distributed Data Parallel replicates the model on each GPU; each GPU gets a different mini-batch. After backward, AllReduce averages gradients across all GPUs so all replicas apply the same gradient update. This makes DDP equivalent to training with a single GPU but with a mini-batch N times larger (where N is the number of GPUs). Communication happens via NCCL ring-AllReduce.

**Q: When would you use model parallelism instead of data parallelism?**
When the model itself doesn't fit in a single GPU's memory. For models like GPT-4 or LLaMA-70B, even a single copy of parameters exceeds A100 memory. Model parallelism splits layers across GPUs; pipeline parallelism schedules micro-batches to reduce the bubble overhead.

**Q: What is gradient accumulation and why use it?**
Gradient accumulation simulates a large batch by running multiple forward-backward passes before each optimizer step. This is useful when memory constraints limit batch size: 4 steps × batch 32 = effective batch 128. The key requirement is to divide the loss by the accumulation steps so the gradient scale matches training with the full batch.

**Q: What is mixed precision training and what is loss scaling?**
Mixed precision uses fp16 for the forward pass (halving memory, doubling throughput on tensor cores) while keeping fp32 master weights for optimizer updates (for numerical precision). Loss scaling multiplies the loss before backward to prevent fp16 gradients from underflowing to zero, then divides before the optimizer step.

## Key Takeaways
- DDP: replicate model, different data per GPU, AllReduce gradients → equivalent to N× larger batch
- Model parallelism: for models too large for one GPU — split layers or tensors across GPUs
- GPU memory = params + grads + optimizer states + activations; Adam doubles memory vs SGD
- Gradient accumulation: divide loss by accumulate_steps; simulate large batch under memory constraint
- Mixed precision: fp16 forward + fp32 optimizer; requires loss scaling to prevent gradient underflow
- Always checkpoint: model state + optimizer state + scheduler + epoch + RNG state for exact resume